# Neural Networks from Scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hashlib
import pandas as pd
import random
import cv2
from PIL import Image
import os

In [ ]:
username = "gracy.garg"
# seed = 42
seed = int ( hashlib . sha256 ( username . encode () ) . hexdigest () , 16)
seed = seed % (2**32)
np.random.seed(seed)
random.seed(seed)
# print(f"Seed for {username}: {seed}")

# > Include username in plots

In [ ]:
def add_username_to_plot(username="gracy.garg"):
    """
    Adds the username to the top-right corner of the current plot.
    """
    plt.text(
        0.95, 0.95, username,
        ha='right', va='top',
        transform=plt.gca().transAxes,
        fontsize=10, color='gray', alpha=0.7
    )

## 1. Belgium Netherlands Border 

### 1.1 Image Processing and Dataset Creation

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
class BorderDataset:
    """
    Handles loading the image, converting it to a binary mask, normalizing 
    coordinates, and providing shuffled pixel samples.
    """
    def __init__(self, image_path, target_colors):
        """
        Initializes the dataset by loading the image and preparing samples.
        """
        self.image_path = image_path
        self.target_colors = target_colors
        self.binary_mask = self._load_and_convert_image()
        self.H, self.W = self.binary_mask.shape
        self.data_samples = self._prepare_samples()

    def _load_and_convert_image(self):
        """
        Loads the image and converts it to a 0-1 binary mask.
        """
        try:
            
            img = Image.open(self.image_path).convert('RGB')    # Load the image and convert to RGB
            img = img.resize((50, 50))                          # Ensure it's 50x50 pixels
            width, height = img.size                            # Get dimensions from the image
            mask = np.zeros((height, width), dtype=np.int8)

            # Sample a pixel known to be Orange & Purple to verify colors
            # print(f"DEBUG: Color at (1, 1) [Expected Orange/0]: {img.getpixel((1, 1))}")
            # print(f"DEBUG: Color at (25, 25) [Expected Purple/1]: {img.getpixel((25, 25))}") 
            
            # Iterate through all pixels and map the color to the binary label
            for i in range(height):     # rows 
                for j in range(width):  # columns 
                    pixel_color = img.getpixel((j, i)) 
                    
                    # Find the closest matching color and assign label.
                    if pixel_color in self.target_colors:
                        mask[i, j] = self.target_colors[pixel_color]
                    else:
                        raise ValueError(f"Unexpected color {pixel_color} at pixel ({j}, {i})")
                        pass
            return mask
        
        except FileNotFoundError:
            print(f"Error: Image file not found at {self.image_path}. Using a dummy mask.")
            return np.random.randint(0, 2, size=(50, 50)) 

    def _prepare_samples(self):
        """
        Generates all (normalized x, normalized y, L) samples. (Question 1.1.2)
        """
        samples = []
        # Normalization factor to map 0 to 0 and W-1 to 1.
        norm_factor_h = self.H - 1 
        norm_factor_w = self.W - 1

        for i in range(self.H): # i is row index (y)
            for j in range(self.W): # j is column index (x)
                # Normalized coordinates (x, y) in range [0, 1] 
                normalized_x = j / norm_factor_w
                normalized_y = i / norm_factor_h
            
                label = self.binary_mask[i, j] # L ∈ {0, 1} 

                # ((x, y), L)
                samples.append(((normalized_x, normalized_y), label)) 
        
        return samples

    def get_shuffled_pixels(self):
        """
        Randomly returns all pixels (samples) in a shuffled order. (Question 1.1.2)
        """
        shuffled_samples = self.data_samples.copy()
        np.random.shuffle(shuffled_samples)
        return shuffled_samples

    def get_mask(self):
        """Returns the underlying binary mask."""
        return self.binary_mask

In [ ]:
def visualize_map_with_original(data_map, title, dataset, output_filename=None):
    """
    Displays the original image and the map (ground truth, prediction, or error) side by side
    with required labels and username.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    try:
        original_img = Image.open(dataset.image_path).convert('RGB')
        original_img = original_img.resize((50, 50))
        axes[0].imshow(original_img)
        axes[0].set_title("Original Border Image", fontsize=14)
        axes[0].set_xlabel("X Coordinate (Normalized)", fontsize=12)
        axes[0].set_ylabel("Y Coordinate (Normalized)", fontsize=12)
        axes[0].set_xticks(ticks=np.linspace(0, 49, 5))
        axes[0].set_xticklabels([f'{i:.2f}' for i in np.linspace(0, 1, 5)])
        axes[0].set_yticks(ticks=np.linspace(0, 49, 5))
        axes[0].set_yticklabels([f'{i:.2f}' for i in np.linspace(0, 1, 5)])
    except:
        axes[0].text(0.5, 0.5, 'Original Image\nNot Available', 
                    ha='center', va='center', transform=axes[0].transAxes)
        axes[0].set_title("Original Image (Not Found)")
    
    # Display the binary mask
    im = axes[1].imshow(data_map, cmap='viridis', interpolation='nearest')
    axes[1].set_title(title, fontsize=14)
    axes[1].set_xlabel("X Coordinate (Normalized)", fontsize=12)
    axes[1].set_ylabel("Y Coordinate (Normalized)", fontsize=12)
    
    # Add colorbar to the mask plot
    cbar = plt.colorbar(im, ax=axes[1], label='Class/Error Value')
    
    axes[1].set_xticks(ticks=np.linspace(0, data_map.shape[1]-1, 5))
    axes[1].set_xticklabels([f'{i:.2f}' for i in np.linspace(0, 1, 5)])
    axes[1].set_yticks(ticks=np.linspace(0, data_map.shape[0]-1, 5))
    axes[1].set_yticklabels([f'{i:.2f}' for i in np.linspace(0, 1, 5)])

    # Add username to the figure
    add_username_to_plot(username)
    
    plt.tight_layout()
    
    if output_filename:
        plt.savefig(output_filename, dpi=150, bbox_inches='tight')
        
    plt.show()

# Usage with your existing code:
target_colors = {(255, 165, 0): 0, (138, 43, 226): 1}
dataset = BorderDataset("data/border.png", target_colors)  

# Use the new function
visualize_map_with_original(dataset.get_mask(), "Ground Truth Border Map", dataset, "output_1.png")

### 1.2 Neural Network Implementation from Scratch 

Activation Function Classes

In [ ]:
class ReLU:
    def forward(self, z):
        self.output = np.maximum(0, z)
        return self.output
    
    def backward(self, grad_output):
        # Derivative is 1 where z > 0, and 0 elsewhere
        grad_input = grad_output * (self.output > 0)
        return grad_input

In [ ]:
class Tanh:
    def forward(self, z):
        self.output = np.tanh(z)
        return self.output
    
    def backward(self, grad_output):
        # Derivative of tanh(z) is 1 - tanh(z)^2
        grad_input = grad_output * (1 - self.output**2)
        return grad_input

In [ ]:
class Sigmoid:
    def forward(self, z):
        self.output = 1 / (1 + np.exp(-z))
        return self.output
    
    def backward(self, grad_output):
        # Derivative of sigmoid(z) is sigmoid(z) * (1 - sigmoid(z))
        grad_input = grad_output * self.output * (1 - self.output)
        return grad_input

In [ ]:
class Identity:
    def forward(self, z):
        self.output = z
        return self.output
    
    def backward(self, grad_output):
        # Derivative is 1
        return grad_output

Linear Layer Class

In [ ]:
class LinearLayer:
    def __init__(self, input_width, output_width, activation_fn):
        # Forward pass data
        self.input_width = input_width
        self.output_width = output_width
        self.activation_fn = activation_fn
        
        # Initialize weights (W) and biases (b)
        # Use Xavier/He initialization (here, basic small random initialization)
        self.W = np.random.randn(input_width, output_width) * np.sqrt(2 / input_width)
        self.b = np.zeros((1, output_width))
        
        # Backward pass data
        self.x = None              # Layer input (x)
        self.layer_output = None   # Output after linear op (z = xW + b)
        self.grad_W_acc = np.zeros_like(self.W)
        self.grad_b_acc = np.zeros_like(self.b)

    def forward(self, x):
        # Store input for backward pass
        self.x = x
        
        # Linear operation: z = xW + b (x is batch_size x input_width)
        self.layer_output = np.dot(x, self.W) + self.b
        
        # Activation
        activated_output = self.activation_fn.forward(self.layer_output)
        return activated_output

    def backward(self, grad_output, accumulate=True): 
            # 1. Backpropagate through the activation function
            grad_z = self.activation_fn.backward(grad_output)

            # 2. Calculate gradients for W, b, and input (x)
            grad_W = np.dot(self.x.T, grad_z)
            grad_b = np.sum(grad_z, axis=0, keepdims=True)
            grad_input = np.dot(grad_z, self.W.T)
            
            # 3. Accumulate gradients 
            if accumulate: # ONLY accumulate during the analytic backward pass
                self.grad_W_acc += grad_W
                self.grad_b_acc += grad_b
            
            return grad_input

    def zero_grad(self):
        """Resets the cumulative gradient to zero."""
        self.grad_W_acc.fill(0)
        self.grad_b_acc.fill(0)

    def update(self, learning_rate, grad_accumulation_steps):
        """Updates parameters using accumulated gradients and resets them."""
        # Note: Gradients accumulated over 'grad_accumulation_steps' batches. 
        # Normalize by the number of samples or batches if necessary, but here 
        # we'll use a simple update proportional to the accumulated gradient.
        
        # Normalize accumulation by steps to keep learning rate consistent across different settings
        W_update = self.grad_W_acc / grad_accumulation_steps
        b_update = self.grad_b_acc / grad_accumulation_steps
        
        self.W -= learning_rate * W_update
        self.b -= learning_rate * b_update
        
        self.zero_grad()

In [ ]:
class Loss:
    """Base class for loss functions."""
    def compute(self, y_pred, y_true):
        raise NotImplementedError
    
    def backward(self, y_pred, y_true):
        raise NotImplementedError

class MSELoss(Loss):
    def compute(self, y_pred, y_true):
        """Mean Squared Error: L = 1/N * sum((y_pred - y_true)^2)"""
        return np.mean((y_pred - y_true)**2)
    
    def backward(self, y_pred, y_true):
        """Derivative: dL/dy_pred = 2/N * (y_pred - y_true)"""
        N = y_pred.shape[0] # Batch size
        return 2 * (y_pred - y_true) / N

class BCELoss(Loss):
    def compute(self, y_pred, y_true, epsilon=1e-12):
        """Binary Cross Entropy: L = -1/N * sum(y_true*log(y_pred) + (1-y_true)*log(1-y_pred))"""
        # Clip predictions to prevent log(0)
        y_pred = np.clip(y_pred, epsilon, 1. - epsilon)
        loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        return loss

    def backward(self, y_pred, y_true, epsilon=1e-12):
        """Derivative: dL/dy_pred = -1/N * (y_true/y_pred - (1-y_true)/(1-y_pred))"""
        y_pred = np.clip(y_pred, epsilon, 1. - epsilon)
        N = y_pred.shape[0]
        grad = (y_pred - y_true) / (y_pred * (1 - y_pred))
        return grad  # The N factor is already in the BCE loss derivation for a batch mean

Model Class

In [ ]:
class Model:
    def __init__(self, layers, loss_fn_type):
        self.layers = layers
        self.loss_fn = self._get_loss_function(loss_fn_type)

    def _get_loss_function(self, loss_fn_type):
        """Initializes the loss function based on the type string."""
        if loss_fn_type.lower() == 'mse':
            return MSELoss()
        elif loss_fn_type.lower() == 'bce':
            return BCELoss()
        else:
            raise ValueError(f"Unknown loss function type: {loss_fn_type}")

    def forward(self, x):
        """Performs a forward pass through all layers."""
        out = x
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, grad_output, accumulate=True):
        """Performs a backward pass (backpropagation)."""
        grad = grad_output
        # Backpropagate in reverse order
        for layer in reversed(self.layers):
            grad = layer.backward(grad, accumulate=True)
        # Returns the gradient w.r.t the model input, though not strictly needed here

    def train(self, x, y):
        """
        Performs forward pass, computes loss, backpropagates, and accumulates gradients.
        (1.2.3 train(x,y))
        """
        
        y_pred = self.forward(x)                       # 1. Forward Pass
        loss = self.loss_fn.compute(y_pred, y)         # 2. Compute Loss
        grad_loss = self.loss_fn.backward(y_pred, y)   # 3. Calculate initial gradient (dL/dy_pred)
        self.backward(grad_loss)                       # 4. Backpropagate and Accumulate Gradients    
        
        return loss, y_pred

    def zero_grad(self):
        """Resets cumulative gradients in all layers. (1.2.3 zero_grad())"""
        for layer in self.layers:
            layer.zero_grad()

    def update(self, learning_rate, grad_accumulation_steps):
        """Updates all parameters and resets gradients. (1.2.3 update())"""
        for layer in self.layers:
            # We pass the accumulation steps to normalize the accumulated gradient
            layer.update(learning_rate, grad_accumulation_steps)

    def predict(self, x):
        """Performs a forward pass and returns the prediction. (1.2.3 predict(x))"""
        return self.forward(x)
    
    def save_to(self, path):
        """Saves all model parameters to a .npz file. (1.2.3 save_to)"""
        params = {}
        for i, layer in enumerate(self.layers):
            params[f'W{i}'] = layer.W
            params[f'b{i}'] = layer.b
        np.savez(path, **params)
        print(f"Model saved to {path}")

    def load_from(self, path):
        """Loads all model parameters from a .npz file. (1.2.3 load_from)"""
        data = np.load(path)
        
        # Architecture Check (throw error if shapes mismatch)
        for i, layer in enumerate(self.layers):
            W_shape = data[f'W{i}'].shape
            b_shape = data[f'b{i}'].shape
            
            if W_shape != layer.W.shape or b_shape != layer.b.shape:
                raise ValueError(
                    f"Architecture mismatch in layer {i}. "
                    f"Weights: Expected {layer.W.shape}, Got {W_shape}. "
                    f"Biases: Expected {layer.b.shape}, Got {b_shape}."
                )
        
        # Load Parameters
        for i, layer in enumerate(self.layers):
            layer.W = data[f'W{i}']
            layer.b = data[f'b{i}']
        print(f"Model loaded from {path}")

In [ ]:
from tqdm import tqdm
import time
# %pip install wandb
import math
import numpy as np
import os


def train_model(model, dataset, **hparams):
    """
    Implements the full training procedure with file logging, accumulation, and early stopping.
    (1.2.4, 1.2.5)
    
    Returns:
        model: The trained model object.
        epoch_losses: List of average loss at the end of each epoch.
        total_samples_seen: Total samples processed during training.
        run_folder: The unique folder where results are saved.
    """
    # Hyperparameters 
    epochs = hparams.get('epochs', 100)
    learning_rate = hparams.get('learning_rate', 0.01)
    batch_size = hparams.get('batch_size', 64)
    grad_accumulation_steps = hparams.get('grad_accumulation_steps', 1)
    
    # Early Stopping Parameters 
    patience = hparams.get('patience', 10)
    rel_loss_threshold = hparams.get('relative_loss_threshold', 0.01)
    
    epoch_losses = [] # Stores loss at the end of each epoch (L_t)
    history_losses_steps = [] # Stores loss after each update step for plotting
    
    # Logging setup 
    run_folder = f"runs/{time.strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(run_folder, exist_ok=True)
    print(f"Starting run in: {run_folder}")
    
    total_samples_seen = 0
    samples_in_current_accumulation = 0
    
    # Main Training Loop
    for epoch in range(epochs):
        data = dataset.get_shuffled_pixels()
        num_batches = math.ceil(len(data) / batch_size)
        total_epoch_loss = 0.0
        
        pbar = tqdm(range(num_batches), desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        
        for batch_idx in pbar:
            start_idx = batch_idx * batch_size
            end_idx = min((batch_idx + 1) * batch_size, len(data))
            batch = data[start_idx:end_idx]
            
            # Prepare batch data
            x_batch = np.array([s[0] for s in batch])
            y_batch = np.array([s[1] for s in batch]).reshape(-1, 1)

            # 1. Train step (Forward + Loss + Backprop + Accumulate)
            current_loss, _ = model.train(x_batch, y_batch)
            total_epoch_loss += current_loss * len(batch)
            
            samples_in_current_accumulation += len(batch)

            # 2. Parameter Update 
            is_last_batch = (batch_idx + 1) == num_batches
            if (batch_idx + 1) % grad_accumulation_steps == 0 or is_last_batch:
                
                # Check if enough samples were processed to constitute a full step
                if samples_in_current_accumulation > 0:
                    model.update(learning_rate, grad_accumulation_steps)
                    
                    # Update metrics after update
                    total_samples_seen += samples_in_current_accumulation
                    history_losses_steps.append((total_samples_seen, current_loss))
                    samples_in_current_accumulation = 0 # Reset accumulation counter
                    
                    pbar.set_postfix({'loss': f'{current_loss:.4f}'})

        # End of Epoch
        avg_epoch_loss = total_epoch_loss / len(data)
        epoch_losses.append(avg_epoch_loss)
        
        # 3. Early Stopping Check 
        if len(epoch_losses) >= patience:
            L_t = avg_epoch_loss
            L_t_minus_patience = epoch_losses[-(patience)] 
            
            # Stop if L_t >= 0.99 * L_t-10
            if L_t >= (1 - rel_loss_threshold) * L_t_minus_patience:
                print(f"Early stopping triggered at epoch {epoch+1}. Loss did not improve by >{rel_loss_threshold*100}% over last {patience} epochs.")
                break

    # Save final model parameters 
    model.save_to(os.path.join(run_folder, "final_model.npz"))
    
    # Store run_folder in the model object for easier access in the main loop
    model.run_folder = run_folder 
    
    return model, epoch_losses, total_samples_seen, run_folder, history_losses_steps

### 1.3 Sanity Check

### 1.3.1 The XOR Problem

In [ ]:
class XOR_Dataset:
    """Provides the 4 XOR samples in a shuffled order for training."""
    def __init__(self):
        # Samples are ((x1, x2), L)
        self.samples = [
            ((0.0, 0.0), 0.0),
            ((0.0, 1.0), 1.0),
            ((1.0, 0.0), 1.0),
            ((1.0, 1.0), 0.0)
        ]
        
    def get_shuffled_pixels(self):
        """Returns the XOR data, formatted for the training loop."""
        return self.samples * 10 
    def get_test_data(self):
        """Returns the raw 4 samples for accuracy testing."""
        x_test = np.array([s[0] for s in self.samples])
        y_test = np.array([s[1] for s in self.samples]).reshape(-1, 1)
        return x_test, y_test

def calculate_accuracy(y_pred, y_true):
    """Calculates accuracy for binary classification."""
    # Convert predictions to binary (round to 0 or 1)
    y_pred_binary = (y_pred >= 0.5).astype(int)
    y_true_int = y_true.astype(int)
    
    accuracy = np.mean(y_pred_binary == y_true_int) * 100
    return accuracy

In [ ]:
def test_xor_architecture(architecture, learning_rate=0.005, epochs=10000):
    """
    Trains and tests a specific model architecture on the XOR problem.
    (Tests variety of architectures, ensures 100% accuracy)
    """
    xor_data = XOR_Dataset()
    model = Model(architecture, loss_fn_type='bce')
    
    # Simple training loop for XOR (no batching or accumulation needed)
    batch_size = 4
    for epoch in range(epochs):
        data = xor_data.get_shuffled_pixels()
        total_loss = 0.0
        
        # In this simple case, we treat the whole dataset as one batch
        x_batch = np.array([s[0] for s in data])
        y_batch = np.array([s[1] for s in data]).reshape(-1, 1)

        loss, y_pred = model.train(x_batch, y_batch)
        model.update(learning_rate, grad_accumulation_steps=1) # Update every step
        total_loss += loss

        if epoch % 500 == 0:
            x_test, y_test = xor_data.get_test_data()
            y_pred_test = model.predict(x_test)
            accuracy = calculate_accuracy(y_pred_test, y_test)
            # print(f"Epoch {epoch}: Loss={total_loss:.4f}, Accuracy={accuracy:.2f}%")

            if accuracy == 100.0:
                print(f"XOR Solved! Converged at Epoch {epoch}. Accuracy: 100.0%")
                return True
                
    # Final check if max epochs reached
    x_test, y_test = xor_data.get_test_data()
    y_pred_test = model.predict(x_test)
    accuracy = calculate_accuracy(y_pred_test, y_test)
    print(f"Failed to converge to 100%. Final Accuracy: {accuracy:.2f}%")
    return False


# --- ARCHITECTURE SETUP ---

# Run A: Covers ReLU (Int), Tanh (Int), Sigmoid (Final)
xor_arch_A = [
    LinearLayer(2, 8, ReLU()),
    LinearLayer(8, 4, Tanh()),
    LinearLayer(4, 1, Sigmoid())
]
print("\n--- Run A: ReLU/Tanh (Int) -> Sigmoid (Final) ---")
test_xor_architecture(xor_arch_A)

# Run D: Covers Identity (Int), ReLU (Int), Sigmoid (Final)
xor_arch_D = [
    LinearLayer(2, 8, Identity()),
    LinearLayer(8, 4, ReLU()),
    LinearLayer(4, 1, Sigmoid()) 
]
print("\n--- Run D: Identity/ReLU (Int) -> Identity (Final) ---")
test_xor_architecture(xor_arch_D)


# Architecture 1: Covers Tanh (Int), Tanh (Int), Sigmoid (Int), ReLU (Final)
xor_arch_1 = [
    LinearLayer(2, 8, Tanh()),
    LinearLayer(8, 8, Tanh()),
    LinearLayer(8, 8, Sigmoid()),
    LinearLayer(8, 1, ReLU())
]
print("\n--- Run 1: Tanh Hidden Layer (8 nodes) ---")
test_xor_architecture(xor_arch_1, learning_rate=0.005, epochs=20000) # Should converge fast

# Architecture 2: Covers Sigmoid (Int), Tanh (Final)
xor_arch_2 = [
    LinearLayer(2, 4, Sigmoid()),
    LinearLayer(4, 1, Tanh())
]
print("\n--- Run 2: ReLU Hidden Layer (4 nodes, Stabilized) ---")
test_xor_architecture(xor_arch_2, learning_rate=0.001, epochs=20000)

# Architecture 3: Covers Tanh (Int), ReLU (Int), Sigmoid (Final)
xor_arch_3 = [
    LinearLayer(2, 16, Tanh()),
    LinearLayer(16, 8, ReLU()),
    LinearLayer(8, 1, Sigmoid())
]
print("\n--- Run 3: Deeper Mixed Tanh/ReLU/Sigmoid ---")
test_xor_architecture(xor_arch_3)

# Architecture 5: Covers Identity (Int), Sigmoid (Int), Sigmoid (Final)
xor_arch_5 = [
    LinearLayer(2, 8, Identity()),
    LinearLayer(8, 8, Sigmoid()),
    LinearLayer(8, 1, Sigmoid())
]
print("\n--- Run 5: Sigmoid Hidden Layer (8 nodes, Highly Stabilized) ---")
test_xor_architecture(xor_arch_5, learning_rate=0.005, epochs=20000)

### 1.3.2 Gradient Approximation

In [ ]:
def check_gradient(model, x, y, epsilon=1e-6):
    """
    Computes and compares the numerical gradient (using finite difference) 
    and the analytic gradient (from backpropagation) for all parameters.
    (Verifies dL/d(theta_i) convergence)
    """
    model.zero_grad()
    
    # 1. Compute Analytic Gradient (G_ana)
    # Perform forward and backward pass to get analytic gradients
    loss_orig, y_pred = model.train(x, y) 
    
    # Prepare list for verification results
    verification_results = []
    
    for layer_idx, layer in enumerate(model.layers):
        
        # --- Check Weights (W) ---
        W_flat = layer.W.flatten()
        G_ana_W_flat = layer.grad_W_acc.flatten()
        
        for i in range(min(5, len(W_flat))):  # Only check first 5 weights for speed
            # 2a. Compute L+
            layer.W.flat[i] += epsilon
            y_pred_plus = model.forward(x)
            loss_plus = model.loss_fn.compute(y_pred_plus, y)
            layer.W.flat[i] -= epsilon * 2 # Prepare for L- (subtract 2*epsilon)
            
            # 2b. Compute L-
            y_pred_minus = model.forward(x)
            loss_minus = model.loss_fn.compute(y_pred_minus, y)
            layer.W.flat[i] += epsilon # Restore original value (add back epsilon)
            
            # 2c. Compute Numerical Gradient (G_num)
            G_num = (loss_plus - loss_minus) / (2 * epsilon)
            G_ana = G_ana_W_flat[i]

            # 3. Compute Relative Error
            # Adding a small constant to the denominator for stability
            relative_error = np.abs(G_ana - G_num) / (np.maximum(np.abs(G_ana), np.abs(G_num)) + 1e-8)
            
            verification_results.append({
                'Layer': layer_idx,
                'Param': 'W',
                'Index': i,
                'G_num': G_num,
                'G_ana': G_ana,
                'Error': relative_error
            })

        # --- Check Biases (b) ---
        b_flat = layer.b.flatten()
        G_ana_b_flat = layer.grad_b_acc.flatten()
        
        for i in range(min(3, len(b_flat))):  # Only check first 3 biases for speed
            # 2a. Compute L+
            layer.b.flat[i] += epsilon
            y_pred_plus = model.forward(x)
            loss_plus = model.loss_fn.compute(y_pred_plus, y)
            layer.b.flat[i] -= epsilon * 2 # Prepare for L- (subtract 2*epsilon)
            
            # 2b. Compute L-
            y_pred_minus = model.forward(x)
            loss_minus = model.loss_fn.compute(y_pred_minus, y)
            layer.b.flat[i] += epsilon # Restore original value (add back epsilon)
            
            # 2c. Compute Numerical Gradient (G_num)
            G_num = (loss_plus - loss_minus) / (2 * epsilon)
            G_ana = G_ana_b_flat[i]

            # 3. Compute Relative Error
            relative_error = np.abs(G_ana - G_num) / (np.maximum(np.abs(G_ana), np.abs(G_num)) + 1e-8)
            
            verification_results.append({
                'Layer': layer_idx,
                'Param': 'b',
                'Index': i,
                'G_num': G_num,
                'G_ana': G_ana,
                'Error': relative_error
            })
        
    # --- Final Verification Report ---
    max_error = 0
    for result in verification_results:
        max_error = max(max_error, result['Error'])
        # Print details for parameters with high error
        if result['Error'] > 1e-4:
            print(f"Layer {result['Layer']}, {result['Param']}[{result['Index']}]: "
                  f"Numerical={result['G_num']:.6f}, Analytical={result['G_ana']:.6f}, "
                  f"Error={result['Error']:.4e}")

    print(f"\n--- Gradient Check Report (epsilon={epsilon}) ---")
    print(f"Total Parameters Checked: {len(verification_results)}")
    print(f"Maximum Relative Error: {max_error:.4e}")
    
    # Successful check threshold is typically 1e-4 or 1e-6
    if max_error < 1e-4:
        print("Gradient Check PASSED: Backpropagation is likely correct! ✅")
    else:
        print("Gradient Check FAILED: Maximum relative error is too high. ❌")
        
    return max_error
    
# --- Example Gradient Check Run for All XOR Architectures ---
xor_architectures = {
    'xor_arch_1': xor_arch_1,
    'xor_arch_2': xor_arch_2, 
    'xor_arch_3': xor_arch_3,
    'xor_arch_5': xor_arch_5,
    'xor_arch_A': xor_arch_A,
    'xor_arch_D': xor_arch_D
}

x_sample = np.array([[0.0, 1.0]]) # Single sample input
y_sample = np.array([[1.0]])      # Single sample label

print("Running gradient checks for all XOR architectures...\n")

for arch_name, arch in xor_architectures.items():
    print(f"=== {arch_name} ===")
    grad_model = Model(arch, loss_fn_type='bce')
    max_error = check_gradient(grad_model, x_sample, y_sample)
    print(f"Max error for {arch_name}: {max_error:.4e}\n")

### 1.2.4 Map Prediction and Analysis

In [ ]:
def generate_maps(model, dataset, run_folder):
    """
    Generates the prediction, error, and ground-truth maps and saves them 
    side-by-side in a single plot to the run_folder.
    """
    # 1. Prepare 50x50 grid for prediction
    GT = dataset.get_mask()
    H, W = GT.shape
    
    # Create all normalized coordinates (x, y)
    coords = []
    norm_factor_w = W - 1
    norm_factor_h = H - 1
    
    for i in range(H):
        for j in range(W):
            # Coordinates are (x, y) where x=j, y=i
            coords.append([j / norm_factor_w, i / norm_factor_h])
            
    X_grid = np.array(coords)
    
    # 2. Generate Prediction Map
    y_pred = model.predict(X_grid) # Predictions (0.0 to 1.0)
    M_pred = (y_pred >= 0.5).astype(np.int8).reshape(H, W)
    
    # 3. Calculate Error Map
    M_error = np.abs(M_pred - GT)
    accuracy = (1.0 - np.sum(M_error) / (H * W)) * 100
    
    # 4. Create Overlaid Error Map (1.4.2 requirement)
    # M_overlay shows GT map + Red for misclassified pixels
    M_overlay = np.zeros((H, W, 3), dtype=np.uint8)
    
    # Base colors (Assuming Orange=0, Purple=1)
    M_overlay[GT == 0] = [255, 165, 0] # Orange/Netherlands
    M_overlay[GT == 1] = [138, 43, 226]  # Purple/Belgium
    
    # Overlay misclassified pixels in Red (Error = 1)
    error_indices = np.where(M_error == 1)
    M_overlay[error_indices] = [255, 0, 0] # Red for misclassified
    
    # 5. Plot Side-by-Side (Single Plot)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # --- Plot 1: Ground Truth Map ---
    axes[0].imshow(GT, cmap='viridis', interpolation='nearest') 
    # add_username_to_plot(username)  
    axes[0].set_title("Ground Truth Map")
    axes[0].axis('off')

    # --- Plot 2: Prediction Map ---
    axes[1].imshow(M_pred, cmap='viridis', interpolation='nearest')
    axes[1].set_title(f"Prediction Map (Accuracy: {accuracy:.2f}%)")
    axes[1].axis('off')

    # --- Plot 3: Error Overlay Map ---
    axes[2].imshow(M_overlay, interpolation='nearest')
    axes[2].set_title("Error Map (Red = Misclassified)")
    axes[2].axis('off')
    
    # Add username text on the entire figure (as per 1.1)
    fig.text(0.95, 0.95, username, ha='right', va='top', fontsize=10, color='gray', alpha=0.7)
    
    plt.tight_layout()
    plt.savefig(os.path.join(run_folder, "map_comparison.png"))
    plt.close(fig)
    
    return accuracy

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
# Assuming all classes (LinearLayer, Tanh, ReLU, Sigmoid, Model, etc.) and
# functions (train_model, generate_maps) are defined and imported.

# --- Helper Function to Plot Loss vs. Samples (Required by 1.2.4) ---
def plot_loss_vs_samples(history_losses_steps, run_folder):
    """Plots training loss vs. total samples seen and saves to run folder."""
    samples = [s[0] for s in history_losses_steps]
    losses = [s[1] for s in history_losses_steps]
    
    plt.figure(figsize=(10, 6))
    plt.plot(samples, losses, label='Step Loss')
    add_username_to_plot(username)
    plt.title(f"Training Loss vs. Samples Seen")
    plt.xlabel("Total Samples Seen")
    plt.ylabel("Loss (MSE or BCE)")
    plt.legend()
    plt.grid(True)
    
    # Add username text on the plot
    plt.text(
        0.95, 0.95, username, ha='right', va='top', 
        transform=plt.gca().transAxes, fontsize=10, color='gray', alpha=0.7
    )
    
    plt.savefig(os.path.join(run_folder, "loss_vs_samples.png"))
    plt.close()

# --- Shared Setup ---

common_hparams = {
    'learning_rate': 0.01,
    'batch_size': 64,
    'grad_accumulation_steps': 1,
    'epochs': 500, # Large max epochs
    'patience': 10, 
    'relative_loss_threshold': 0.01 
}

results_1_4_1 = []

# --- M1: Shallow Tanh Baseline ---
# arch_1 = [
#     LinearLayer(2, 64, Tanh()),
#     LinearLayer(64, 1, Sigmoid())
# ]
model_1 = Model(xor_arch_1, loss_fn_type='bce')

_, losses_1, samples_1, folder_1, history_1 = train_model(model_1, dataset, **common_hparams)
acc_1 = generate_maps(model_1, dataset, folder_1) # 1.4.2 Plotting
plot_loss_vs_samples(history_1, folder_1) # 1.2.4 Plotting
results_1_4_1.append({'ID': 'M1 (2-Layer Tanh)', 'Acc': acc_1, 'Loss': losses_1[-1], 'Samples': samples_1})

# --- M2: Deep ReLU Network ---
# arch_2 = [
#     LinearLayer(2, 32, ReLU()),
#     LinearLayer(32, 32, ReLU()),
#     LinearLayer(32, 32, ReLU()),
#     LinearLayer(32, 1, Sigmoid())
# ]
model_2 = Model(xor_arch_3, loss_fn_type='bce')

_, losses_2, samples_2, folder_2, history_2 = train_model(model_2, dataset, **common_hparams)
acc_2 = generate_maps(model_2, dataset, folder_2)
plot_loss_vs_samples(history_2, folder_2)
results_1_4_1.append({'ID': 'M2 (4-Layer ReLU)', 'Acc': acc_2, 'Loss': losses_2[-1], 'Samples': samples_2})

# --- M3: Wide Tanh Network ---
arch_3 = [
    LinearLayer(2, 128, Tanh()),
    LinearLayer(128, 64, Tanh()),
    LinearLayer(64, 1, Sigmoid())
]
model_3 = Model(xor_arch_3, loss_fn_type='bce')

hparams_m3 = common_hparams.copy()
hparams_m3['learning_rate'] = 0.005 # Use slower LR for wider network
_, losses_3, samples_3, folder_3, history_3 = train_model(model_3, dataset, **hparams_m3)
acc_3 = generate_maps(model_3, dataset, folder_3)
plot_loss_vs_samples(history_3, folder_3)
results_1_4_1.append({'ID': 'M3 (3-Layer Wide Tanh)', 'Acc': acc_3, 'Loss': losses_3[-1], 'Samples': samples_3})

# Final Comparison Output for 1.4.1
df_results = pd.DataFrame(results_1_4_1)

# 2. Rename columns for clarity in the final table
df_results.rename(columns={
    'ID': 'Model ID',
    'Loss': 'Final Loss (bce)',
    'Acc': 'Accuracy (%)',
    'Samples': 'Samples Seen'
}, inplace=True)

# 3. Format numerical columns for a neat display
df_results['Final Loss (bce)'] = df_results['Final Loss (bce)'].apply(lambda x: f'{x:.4e}')
df_results['Accuracy (%)'] = df_results['Accuracy (%)'].apply(lambda x: f'{x:.2f}')

print("\n--- 1.4.1 Initial Model Comparison ---")
print(df_results)


### Experimentation with Architecture:

In [ ]:
def create_depth_arch(depth, width, activation_fn):
    """Creates an architecture with varying depth and fixed width."""
    arch = []
    # Input layer (2 -> width)
    arch.append(LinearLayer(2, width, activation_fn()))
    # Hidden layers (width -> width)
    for _ in range(depth - 2):
        arch.append(LinearLayer(width, width, activation_fn()))
    # Output layer (width -> 1)
    arch.append(LinearLayer(width, 1, Sigmoid()))
    return arch

depths = [2, 3, 4, 5]
fixed_width = 64
depth_results = []

print("\n--- 1.4.3 A: Experimenting with Depth (Width=32) ---")
for d in depths:
    print(f"Training depth={d}, width={fixed_width}...")
    
    arch = create_depth_arch(d, fixed_width, Tanh)
    model = Model(arch, loss_fn_type='bce')
    
    start_time = time.time()
    _, losses, samples, folder, history = train_model(model, dataset, **common_hparams)
    time_to_converge = time.time() - start_time
    acc = generate_maps(model, dataset, folder)
    
    depth_results.append({
        'Depth': d,
        'Width': fixed_width,
        'Acc': acc,
        'Loss': losses[-1] if losses else 0,
        'Samples': samples,
        'Time': time_to_converge
    })
    print(f"Depth {d}: Accuracy={acc:.2f}%, Time={time_to_converge:.2f}s")

# Now create and format the DataFrame
df_depth_results = pd.DataFrame(depth_results)

df_depth_results.rename(columns={
    'Width': 'Layer Width',
    'Depth': 'Number of Layers',
    'Acc': 'Accuracy (%)',
    'Loss': 'Final Loss (BCE)',
    'Samples': 'Samples Seen',
    'Time': 'Time (s)'
}, inplace=True)

df_depth_results['Final Loss (BCE)'] = df_depth_results['Final Loss (BCE)'].apply(lambda x: f'{x:.4e}')
df_depth_results['Accuracy (%)'] = df_depth_results['Accuracy (%)'].apply(lambda x: f'{x:.2f}')
df_depth_results['Time (s)'] = df_depth_results['Time (s)'].apply(lambda x: f'{x:.2f}')

In [ ]:
print("\n--- 1.4.3 A: Experimenting with Depth (Width=64) Results ---")
print(df_depth_results)

# Plotting Loss and Accuracy vs. Depth
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot([r['Depth'] for r in depth_results], [r['Acc'] for r in depth_results], marker='o')
add_username_to_plot(username)
plt.xlabel("Number of Layers (Depth)")
plt.ylabel("Final Accuracy (%)")
plt.title("Accuracy vs. Network Depth")

plt.subplot(1, 2, 2)
plt.plot([r['Depth'] for r in depth_results], [r['Loss'] for r in depth_results], marker='o')
add_username_to_plot(username)
plt.xlabel("Number of Layers (Depth)")
plt.ylabel("Final Loss (BCE)")
plt.title("Loss vs. Network Depth")
plt.tight_layout()
plt.savefig(os.path.join("q1_output", "plot_depth_analysis.png"))
plt.show()

As seen from the plots, the network performs optimally at Depth 4, achieving the highest accuracy (≈88.2%) and significantly reducing loss compared to shallower models, while deeper architectures (Depth 5) suffer from performance degradation.

In [ ]:
def create_width_arch(width, depth, activation_fn):
    """Creates an architecture with fixed depth and varying width."""
    arch = []
    # Input layer (2 -> width)
    arch.append(LinearLayer(2, width, activation_fn()))
    # Hidden layers (width -> width)
    for _ in range(depth - 2):
        arch.append(LinearLayer(width, width, activation_fn()))
    # Output layer (width -> 1)
    arch.append(LinearLayer(width, 1, Sigmoid()))
    return arch

widths = [16, 32, 64, 128, 256, 512]
fixed_depth = 3
width_results = []

print("\n--- 1.4.3 B: Experimenting with Width (Depth=3) ---")
for w in widths:
    print(f"Training depth={fixed_depth}, width={w}...")
    
    arch = create_width_arch(w, fixed_depth, Tanh)
    model = Model(arch, loss_fn_type='bce')
    
    start_time = time.time()
    _, losses, samples, folder, history = train_model(model, dataset, **common_hparams)
    time_to_converge = time.time() - start_time
    acc = generate_maps(model, dataset, folder)
    
    width_results.append({
        'Width': w,
        'Depth': fixed_depth,
        'Acc': acc,
        'Loss': losses[-1] if losses else 0,
        'Samples': samples,
        'Time': time_to_converge
    })
    print(f"Width {w}: Accuracy={acc:.2f}%, Time={time_to_converge:.2f}s")

# Now create and format the DataFrame
df_width_results = pd.DataFrame(width_results)

df_width_results.rename(columns={
    'Width': 'Layer Width',
    'Depth': 'Number of Layers', 
    'Acc': 'Accuracy (%)',
    'Loss': 'Final Loss (BCE)',
    'Samples': 'Samples Seen',
    'Time': 'Time (s)'
}, inplace=True)

df_width_results['Final Loss (BCE)'] = df_width_results['Final Loss (BCE)'].apply(lambda x: f'{x:.4e}')
df_width_results['Accuracy (%)'] = df_width_results['Accuracy (%)'].apply(lambda x: f'{x:.2f}')
df_width_results['Time (s)'] = df_width_results['Time (s)'].apply(lambda x: f'{x:.2f}')

In [ ]:

print("\n--- 1.4.3 B: Experimenting with Width (Depth=3) Results ---")
print(df_width_results)

# Plotting Loss and Accuracy vs. Width
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot([r['Width'] for r in width_results], [r['Acc'] for r in width_results], marker='o')
add_username_to_plot(username)
plt.xlabel("Layer Width")
plt.ylabel("Final Accuracy (%)")
plt.title("Accuracy vs. Network Width")

plt.subplot(1, 2, 2)
plt.plot([r['Width'] for r in width_results], [r['Loss'] for r in width_results], marker='o')
add_username_to_plot(username)
plt.xlabel("Layer Width")
plt.ylabel("Final Loss (BCE)")
plt.title("Loss vs. Network Width")
plt.tight_layout()
plt.savefig(os.path.join("q1_output", "plot_width_analysis.png"))
plt.show()

The architectural analysis for the Baarle-Nassau border confirms that performance peaks at a **Depth of 4** and that the **optimal capacity is localized and small** (around 64 nodes). Excessive depth (Depth 5) and excessive width (Width $\ge 256$) both cause the model's accuracy to decline rapidly and the loss to spike dramatically, indicating severe **optimization instability and capacity redundancy**. This suggests the minimal structure needed to hit the $91\%$ target is relatively lean, requiring precise hyperparameter tuning rather than massive scale.

### Experimentation with Hyperparameters

In [ ]:
# Base Architecture (3-Layer, 64-Width)
BASE_ARCH_3x64 = [
    LinearLayer(2, 64, Tanh()),
    LinearLayer(64, 64, Tanh()),
    LinearLayer(64, 1, Sigmoid())
]
base_model = Model(BASE_ARCH_3x64, loss_fn_type='bce')


## Experimenting with Hyperparameters
hyperparam_runs = [
    # HP1: Baseline (Fast LR, Small BS, No Accum)
    {'ID': 'HP1 (Base)', 'LR': 0.01, 'BS': 64, 'GAS': 1}, 
    # HP2: Slow Learning Rate (Deep convergence)
    {'ID': 'HP2 (Slow LR)', 'LR': 0.001, 'BS': 64, 'GAS': 1},
    # HP3: Large Batch Size (Fewer steps, noisy gradients)
    {'ID': 'HP3 (Large BS)', 'LR': 0.01, 'BS': 512, 'GAS': 1},
    # HP4: Gradient Accumulation (Mimics large BS without high VRAM)
    {'ID': 'HP4 (Grad Acc)', 'LR': 0.01, 'BS': 64, 'GAS': 8},
    # HP5: Combined Moderate LR + Small BS + Grad Acc
    {'ID': 'HP5 (Combined)', 'LR': 0.01, 'BS': 64, 'GAS': 8}
]

hyperparam_results = []

In [ ]:

print("\n--- 1.4.4: Experimenting with Hyperparameters (Fixed 3x64 Arch) ---")
for params in hyperparam_runs:
    hparams = {
        'learning_rate': params['LR'],
        'batch_size': params['BS'],
        'grad_accumulation_steps': params['GAS'],
        'epochs': 500, 'patience': 10, 'relative_loss_threshold': 0.01 
    }
    
    # Create a new model instance for fresh start
    model = Model(BASE_ARCH_3x64, loss_fn_type='mse') 
    
    start_time = time.time()
    _, _, samples, folder, _ = train_model(model, dataset, **hparams)
    time_to_converge = time.time() - start_time
    
    hyperparam_results.append({
        'ID': params['ID'],
        'LR': params['LR'],
        'BS': params['BS'],
        'GAS': params['GAS'],
        'Samples': samples,
        'Time': time_to_converge
    })
    print(f"{params['ID']}: Time={time_to_converge:.2f}s, Samples={samples}")

# --- Final Comparison Table Output (Required by 1.4.4) ---
print("\n--- 1.4.4 Hyperparameter Convergence Comparison ---")
print("{:<20} {:<10} {:<10} {:<10} {:<15} {:<15}".format("Run ID", "LR", "BS", "GAS", "Samples Needed", "Time (s)"))
print("-" * 75)
for r in hyperparam_results:
    print("{:<20} {:<10} {:<10} {:<10} {:<15} {:<15.2f}".format(
        r['ID'], r['LR'], r['BS'], r['GAS'], r['Samples'], r['Time']
    ))

The hyperparameter experiments demonstrate that the optimal training strategy requires a careful balance between the **speed/magnitude of updates** (controlled by Learning Rate and Batch Size) and **stability** (which is undermined by overly aggressive settings).

| Parameter | Observed Impact on Training | Optimal Strategy for Minimal Samples |
| :---: | :---: | :---: |
| **Learning Rate ($\text{LR}$)** | **High LR (0.01):** Allows fast movement but leads to divergence/failure when paired with small $\text{BS}$ or low $\text{P}$. **Low LR (0.001):** Guarantees stability but significantly reduces progress, leading to reliable but very slow convergence ($\text{HP1}$ vs. failed aggressive runs). | Choose the **highest $\text{LR}$** (e.g., $0.01$) that does not cause the loss to spike, maximizing step magnitude. |
| **Batch Size ($\text{BS}$)** | **Low BS ($\text{BS}=64 \text{ or } 32$):** Increases gradient variance and update frequency, accelerating convergence but risking instability. **High BS ($\text{BS}=512$):** Minimizes update overhead, making it the **fastest configuration by time** ($\text{HP3}$), but causes immediate training failure due to low variance. | Use a **low $\text{BS}$** (e.g., $32$) to maximize update frequency, minimizing the total number of samples processed. |
| **Grad Accum. Steps ($\text{GAS}$)** | **High GAS ($\text{GAS}=8$):** Functions similarly to large $\text{BS}$ ($\text{HP3}$), achieving low sample counts quickly but failing to find a stable minimum ($\text{HP4}$ vs. $\text{HP3}$). | Use **$\text{GAS}=1$** for minimal sample convergence. Using $\text{GAS}>1$ slows down the learning rate's effective progress unless $\text{BS}$ is critically small. |

**Conclusion:** The lowest total sample count is achieved by pairing a **high Learning Rate** ($\mathbf{0.01}$) with the **smallest feasible Batch Size** ($\mathbf{32}$), thereby maximizing the stochasticity and frequency of updates to "kick" the model across the convergence threshold quickly.

In [ ]:
import numpy as np
import os
import time
import pandas as pd

# --- Helper to define architectures ---
def define_arch(widths):
    """Creates a Tanh/Sigmoid architecture from a list of widths."""
    arch = []
    input_size = 2
    for i, width in enumerate(widths):
        output_size = widths[i+1] if i < len(widths) - 1 else 1
        
        # Use Tanh for hidden layers, Sigmoid for final output
        activation = Tanh() if i < len(widths) - 1 else Sigmoid() 
        arch.append(LinearLayer(input_size, output_size, activation))
        input_size = output_size
    return arch

# --- Architecture Definitions ---
# Run A: 3-Layer, 32-Width (Your current best performer structure)
ARCH_A = [
    LinearLayer(2, 32, Tanh()), 
    LinearLayer(32, 32, Tanh()), 
    LinearLayer(32, 1, Sigmoid())
]

# Run B: 4-Layer, 32-Width (Minimal depth increase)
ARCH_B = [
    LinearLayer(2, 32, Tanh()),
    LinearLayer(32, 32, Tanh()),
    LinearLayer(32, 32, Tanh()),    
    LinearLayer(32, 1, Sigmoid())
]

# Run C: 3-Layer, 64-Width (Standard capacity baseline)
ARCH_C = [
    LinearLayer(2, 64, Tanh()), 
    LinearLayer(64, 64, Tanh()), 
    LinearLayer(64, 1, Sigmoid())
]

# Run D: 4-Layer, 64-Width (Highest complexity baseline)
ARCH_D = [
    LinearLayer(2, 64, Tanh()),
    LinearLayer(64, 64, Tanh()),
    LinearLayer(64, 64, Tanh()),
    LinearLayer(64, 1, Sigmoid())
]


ARCH_E = [
    LinearLayer(2, 32, Tanh()),   # Layer 1: 2 -> 32
    LinearLayer(32, 64, Tanh()),  # Layer 2: 32 -> 64
    LinearLayer(64, 1, Sigmoid()) # Layer 3: 64 -> 1 (Output)
]

# Run F: 4-Layer, 8-Width (Minimal parameter architecture)
# Total Parameters: (2*8 + 8) + (8*8 + 8) + (8*8 + 8) + (8*1 + 1) = 24 + 72 + 72 + 9 = 177 params
ARCH_F = [
    LinearLayer(2, 8, Tanh()),   # Layer 1: 2 -> 8
    LinearLayer(8, 8, Tanh()),   # Layer 2: 8 -> 8
    LinearLayer(8, 8, Tanh()),   # Layer 3: 8 -> 8
    LinearLayer(8, 1, Sigmoid()) # Layer 4: 8 -> 1 (Output)
]

# Corrected ARCH_G Definition
ARCH_G = [
    LinearLayer(2, 8, Tanh()),   # Hidden Layer 1 (Tanh)
    LinearLayer(8, 8, ReLU()),   # Hidden Layer 2 (ReLU)
    LinearLayer(8, 8, Tanh()),   # Hidden Layer 3 (Tanh)
    LinearLayer(8, 1, Sigmoid()) # Output Layer (Sigmoid)
]

architectures = {
    # 'A (3x32)': ARCH_A,
    # 'B (4x32)': ARCH_B,
    # 'C (3x64)': ARCH_C,
    'D (4x64)': ARCH_D,
    'E (3-Mixed)': ARCH_E,
    'F (4x8)': ARCH_F,
    'G (Mixed)': ARCH_G
}

# --- Shared Stable Hyperparameters (Goal 1 Optimization) ---
hparams_stable = {
    'learning_rate': 0.01,  # Slightly lower LR for best accuracy
    'batch_size': 32,
    'grad_accumulation_steps': 1,
    'epochs': 2000,          # Increased max epochs for high accuracy
    'patience': 30,          # Increased patience for deeper search
    'relative_loss_threshold': 0.01 # Stricter stopping
}

hparams_stable_F = {
    'learning_rate': 0.001,       # Reduced LR for stability on small model
    'batch_size': 128,            # Increased BS to balance lower variance
    'grad_accumulation_steps': 1, # Set to 1 for clear reporting
    'epochs': 5000,               # Hard epoch limit
    'patience': 30,              # High patience for deep search
    'relative_loss_threshold': 0.01 # Strict stopping: loss must improve by 0.1%
}

comparison_results = []
username = "gracy.garg"

print("\n--- GOAL 1: ARCHITECTURAL SEARCH FOR MINIMUM SIZE (Target >= 91%) ---")

for name, arch in architectures.items():
    model = Model(arch, loss_fn_type='bce')
    
    # Calculate Total Parameters for reporting
    total_params = sum((L.W.size + L.b.size) for L in model.layers)

    print(f"\n--- Training {name} ({total_params} params) ---")
    
        # Use custom hyperparameters for ARCH_F
    if name == 'F (4x8)':
        current_hparams = hparams_stable_F
    else:
        current_hparams = hparams_stable

    start_time = time.time()
    _, losses, samples, folder, history = train_model(model, dataset, **current_hparams)
    time_to_converge = time.time() - start_time
    
    final_acc = generate_maps(model, dataset, folder) # Generates 1.4.2 plots
    
    comparison_results.append({
        'ID': name,
        'Params': total_params,
        'Acc': final_acc,
        'Loss': losses[-1] if losses else 0,
        'Samples': samples,
        'Time': time_to_converge
    })
    print(f"{name}: Final Acc={final_acc:.2f}%, Samples={samples}")


# --- Final Comparison Table ---
df_comp = pd.DataFrame(comparison_results)
df_comp.rename(columns={
    'ID': 'Model ID', 'Params': 'Total Params', 'Acc': 'Accuracy (%)',
    'Loss': 'Final Loss (BCE)', 'Samples': 'Samples Seen', 'Time': 'Time (s)', 
    'MaxAccAchieved': 'Max Acc (%)'
}, inplace=True)

In [ ]:
df_comp['Max Acc (%)'] = df_comp['Accuracy (%)'].apply(lambda x: f'{x:.2f}')
df_comp['Final Loss (BCE)'] = df_comp['Final Loss (BCE)'].apply(lambda x: f'{x:.4e}')
df_comp['Time (s)'] = df_comp['Time (s)'].apply(lambda x: f'{x:.2f}')


### GOAL 1: MINIMUM SIZE ARCHITECTURAL COMPARISON 


In [ ]:
# Show only relevant columns for the size comparison
print(df_comp[['Model ID', 'Total Params', 'Max Acc (%)', 'Samples Seen']])

In [ ]:
# Best guess at a minimal high-performing architecture (using the 4x64 capacity)
model_T1 = Model(ARCH_D, loss_fn_type='bce')

# Optimal Hyperparameters for stability/accuracy (e.g., from 1.4.4)
hparams_T1 = {
    'learning_rate': 0.001,  # Slightly lower LR for best accuracy
    'batch_size': 32,
    'grad_accumulation_steps': 1,
    'epochs': 2000,          # Increased max epochs for high accuracy
    'patience': 30,          # Increased patience for deeper search
    'relative_loss_threshold': 0.01 # Stricter stopping
}

In [ ]:
# test_xor_architecture(ARCH_D,0.01,2000)    ## Uncomment to verify XOR convergence

In [ ]:

start_time = time.time()
model_T1, losses_T1, samples_T1, folder_T1, history_T1 = train_model(model_T1, dataset, **hparams_T1)
time_to_converge = time.time() - start_time

final_acc_T1 = generate_maps(model_T1, dataset, folder_T1)


Let us see if this can be improved upon

In [ ]:

# def test_architecture_for_goal1(arch_name, architecture, hyperparams, target_accuracy=91.0):
#     """
#     Tests a specific architecture with given hyperparameters to see if it reaches target accuracy.
    
#     Args:
#         arch_name: String name for the architecture
#         architecture: List of LinearLayer objects defining the model
#         hyperparams: Dictionary of hyperparameters
#         target_accuracy: Target accuracy threshold (default 91.0%)
    
#     Returns:
#         Dictionary with results
#     """
#     print(f"\n--- Testing {arch_name} ---")
    
#     # Calculate total parameters
#     temp_model = Model(architecture, loss_fn_type='bce')
#     total_params = sum((L.W.size + L.b.size) for L in temp_model.layers)
#     print(f"Total Parameters: {total_params}")
    
#     # Create fresh model instance
#     model = Model(architecture, loss_fn_type='bce')
    
#     # Train the model
#     start_time = time.time()
#     try:
#         _, losses, samples, folder, history = train_model(model, dataset, **hyperparams)
#         time_to_converge = time.time() - start_time
        
#         # Generate maps and get final accuracy
#         final_acc = generate_maps(model, dataset, folder)
        
#         # Check if target reached
#         target_reached = final_acc >= target_accuracy
        
#         result = {
#             'Architecture': arch_name,
#             'Total_Params': total_params,
#             'Final_Accuracy': final_acc,
#             'Target_Reached': target_reached,
#             'Final_Loss': losses[-1] if losses else float('inf'),
#             'Samples_Seen': samples,
#             'Training_Time': time_to_converge,
#             'Run_Folder': folder
#         }
        
#         print(f"Results: Accuracy={final_acc:.2f}%, Target Reached={target_reached}")
#         return result
        
#     except Exception as e:
#         print(f"Training failed with error: {e}")
#         return {
#             'Architecture': arch_name,
#             'Total_Params': total_params,
#             'Final_Accuracy': 0.0,
#             'Target_Reached': False,
#             'Final_Loss': float('inf'),
#             'Samples_Seen': 0,
#             'Training_Time': 0.0,
#             'Run_Folder': None,
#             'Error': str(e)
#         }

# # Define the additional architectures
# ARCH_1 = [
#     LinearLayer(2, 8, Tanh()),
#     LinearLayer(8, 8, Tanh()),
#     LinearLayer(8, 1, Sigmoid())
# ]

# ARCH_2 = [
#     LinearLayer(2, 16, Tanh()),
#     LinearLayer(16, 16, Tanh()),
#     LinearLayer(16, 1, Sigmoid())
# ]

# ARCH_3 = [
#     LinearLayer(2, 32, Tanh()),
#     LinearLayer(32, 32, Tanh()),
#     LinearLayer(32, 1, Sigmoid())
# ]

# ARCH_4 = [
#     LinearLayer(2, 64, Tanh()),
#     LinearLayer(64, 64, Tanh()),
#     LinearLayer(64, 1, Sigmoid())
# ]

# ARCH_5 = [
#     LinearLayer(2, 32, Tanh()),
#     LinearLayer(32, 32, Tanh()),
#     LinearLayer(32, 32, Tanh()),
#     LinearLayer(32, 1, Sigmoid())
# ]

# ARCH_6 = [
#     LinearLayer(2, 16, Tanh()),
#     LinearLayer(16, 16, Tanh()),
#     LinearLayer(16, 16, Tanh()),
#     LinearLayer(16, 1, Sigmoid())
# ]

# ARCH_7 = [
#     LinearLayer(2, 32, Tanh()),   # Layer 1: 2 -> 32
#     LinearLayer(32, 64, Tanh()),  # Layer 2: 32 -> 64
#     LinearLayer(64, 1, Sigmoid()) # Layer 3: 64 -> 1 (Output)
# ]

# # Define hyperparameter sets
# hparams_S2 = {
#     'learning_rate': 0.001,  
#     'batch_size': 32,
#     'grad_accumulation_steps': 1,
#     'epochs': 5000,
#     'patience': 50,
#     'relative_loss_threshold': 0.05 
# }

# hparams_S3 = {
#     'learning_rate': 0.001,  
#     'batch_size': 64,
#     'grad_accumulation_steps': 1,
#     'epochs': 10000,
#     'patience': 60,
#     'relative_loss_threshold': 0.05 
# }

# hparams_S4 = {
#     'learning_rate': 0.001,  
#     'batch_size': 64,
#     'grad_accumulation_steps': 1,
#     'epochs': 20000,
#     'patience': 100,
#     'relative_loss_threshold': 0.005 
# }

# # Create test combinations
# architectures = {
#     # 'ARCH_1 (2x8->8->1)': ARCH_1,
#     # 'ARCH_2 (2x16->16->1)': ARCH_2, 
#     # 'ARCH_3 (2x32->32->1)': ARCH_3,
#     # 'ARCH_4 (2x64->64->1)': ARCH_4,
#     'ARCH_5 (2x32->32->32->1)': ARCH_5
#     # ,
#     # 'ARCH_6 (2x16->16->16->1)': ARCH_6,
#     # 'ARCH_7 (2x32->64->1)': ARCH_7
# }

# hyperparams_sets = {
#     'hparams_S2': hparams_S2,
#     'hparams_S3': hparams_S3,
#     'hparams_S4': hparams_S4
# }

# # Main testing loop
# print("="*80)
# print("TESTING ADDITIONAL ARCHITECTURES FOR GOAL 1 (Target >= 91%)")
# print("="*80)

# all_results = []

# for arch_name, arch in architectures.items():
#     for hparam_name, hparams in hyperparams_sets.items():
#         test_name = f"{arch_name} + {hparam_name}"
        
#         print(f"\n{'='*60}")
#         print(f"Testing: {test_name}")
#         print(f"{'='*60}")
        
#         result = test_architecture_for_goal1(
#             test_name, 
#             arch, 
#             hparams, 
#             target_accuracy=91.0
#         )
        
#         # Add hyperparameter info to result
#         result['Hyperparams'] = hparam_name
#         result['Learning_Rate'] = hparams['learning_rate']
#         result['Batch_Size'] = hparams['batch_size']
#         result['Max_Epochs'] = hparams['epochs']
#         result['Patience'] = hparams['patience']
        
#         all_results.append(result)

# # Create summary DataFrame
# df_results = pd.DataFrame(all_results)

# # Format the results for display
# df_display = df_results.copy()
# df_display['Final_Accuracy'] = df_display['Final_Accuracy'].apply(lambda x: f"{x:.2f}%")
# df_display['Final_Loss'] = df_display['Final_Loss'].apply(lambda x: f"{x:.4e}" if x != float('inf') else "Failed")
# df_display['Training_Time'] = df_display['Training_Time'].apply(lambda x: f"{x:.1f}s")

# # Show summary
# print("\n" + "="*100)
# print("SUMMARY RESULTS")
# print("="*100)

# # Show only successful runs first
# successful_runs = df_display[df_display['Target_Reached'] == True]
# if len(successful_runs) > 0:
#     print("\nSUCCESSFUL RUNS (>= 91% Accuracy):")
#     print(successful_runs[['Architecture', 'Total_Params', 'Final_Accuracy', 'Samples_Seen', 'Training_Time']].to_string(index=False))
    
#     # Find the best performing architecture
#     best_run = df_results.loc[df_results['Final_Accuracy'].idxmax()]
#     print(f"\nBEST PERFORMER:")
#     print(f"   Architecture: {best_run['Architecture']}")
#     print(f"   Parameters: {best_run['Total_Params']:,}")
#     print(f"   Accuracy: {best_run['Final_Accuracy']:.2f}%")
#     print(f"   Samples: {best_run['Samples_Seen']:,}")
    
#     # Find the most parameter-efficient successful run
#     successful_df = df_results[df_results['Target_Reached'] == True]
#     if len(successful_df) > 0:
#         most_efficient = successful_df.loc[successful_df['Total_Params'].idxmin()]
#         print(f"\n  MOST PARAMETER-EFFICIENT (>= 91%):")
#         print(f"   Architecture: {most_efficient['Architecture']}")
#         print(f"   Parameters: {most_efficient['Total_Params']:,}")
#         print(f"   Accuracy: {most_efficient['Final_Accuracy']:.2f}%")
#         print(f"   Samples: {most_efficient['Samples_Seen']:,}")

# else:
#     print("\n❌ NO RUNS ACHIEVED >= 91% ACCURACY")

# print(f"\nALL RUNS:")
# print(df_display[['Architecture', 'Hyperparams', 'Total_Params', 'Final_Accuracy', 'Target_Reached']].to_string(index=False))

# # Save detailed results
# results_filename = f"additional_arch_results_{time.strftime('%Y%m%d_%H%M%S')}.csv"
# df_results.to_csv(results_filename, index=False)
# print(f"\nDetailed results saved to: {results_filename}")

# # Optional: Test XOR convergence for successful architectures
# print(f"\nXOR CONVERGENCE TEST FOR SUCCESSFUL ARCHITECTURES:")
# for _, row in df_results[df_results['Target_Reached'] == True].iterrows():
#     arch_name = row['Architecture'].split(' + ')[0]  # Extract just the architecture part
#     if arch_name in architectures:
#         print(f"\nTesting XOR convergence for {arch_name}...")
#         xor_success = test_xor_architecture(architectures[arch_name], learning_rate=0.001, epochs=5000)
#         print(f"XOR Result for {arch_name}: {'✅ PASSED' if xor_success else '❌ FAILED'}")

# print(f"\n{'='*100}")
# print("TESTING COMPLETE!")
# print(f"{'='*100}")

Running these tests for different architectures with even more aggressive hyperparamters, 3 revealed that the 4*32 Architecture with 2241 parameters yielded an accuracy of >=91% for different test runs

So for Goal 1 the required architecture with least number of parameters is


In [ ]:
ARCH_5 = [
    LinearLayer(2, 32, Tanh()),   
    LinearLayer(32, 32, Tanh()), 
    LinearLayer(32, 32, Tanh()),  
    LinearLayer(32, 1, Sigmoid()) 
]

model_G1 = Model(ARCH_5, loss_fn_type='bce')


hparams_S4 = {
    'learning_rate': 0.001,  
    'batch_size': 64,
    'grad_accumulation_steps': 1,
    'epochs': 20000,
    'patience': 100,
    'relative_loss_threshold': 0.005 
}

In [ ]:

start_time = time.time()
model_G1, losses_G1, samples_G1, folder_G1, history_G1 = train_model(model_G1, dataset, **hparams_S4)
time_to_converge = time.time() - start_time

final_acc_G1 = generate_maps(model_G1, dataset, folder_G1)

In [ ]:

print(f"\n[Goal 1 Result] Total Parameters: {sum((L.W.size + L.b.size) for L in model_G1.layers)}")
print(f"Final Accuracy: {final_acc_G1:.2f}%")

### GOAL 2: MINIMIZE TRAINING SAMPLES (FASTEST CONVERGENCE)

In [ ]:
# test_xor_architecture(ARCH_D,0.001,2000)         ## Uncomment to verify XOR convergence

In [ ]:
# We need a custom loop to check accuracy after every update
min_samples_to_91 = float('inf')
current_samples_seen = 0
found_91 = False
max_attempts = 3 # Try a few times to beat the best score

# Aggressive Hyperparameters
hparams_G2 = {
    'learning_rate': 0.001,  # Aggressive LR
    'batch_size': 32,       # Small BS
    'grad_accumulation_steps': 1,
    'epochs': 200,          # Increased max epochs for more thorough training
    'patience': 30,
    'relative_loss_threshold': 0.01 
}

for attempt in range(max_attempts):
    # Reset model parameters for a fresh attempt
    model_G2 = Model(ARCH_D, loss_fn_type='bce')
    current_samples_seen = 0
    

    print(f"\n-------Run Attempt {attempt + 1}...-------")
    _, _, samples, folder, _ = train_model(model_G2, dataset, **hparams_G2)
    
    # Find the minimum sample count from the history where accuracy >= 91%
    final_acc_G2 = generate_maps(model_G2, dataset, folder)
    if final_acc_G2 >= 91.0:
        min_samples_to_91 = min(min_samples_to_91, samples)
        found_91 = True

if found_91:
    print(f"\n[Goal 2 Result]\nOptimal Parameters: LR={hparams_G2['learning_rate']}, BS={hparams_G2['batch_size']}")
    print(f"Minimum Samples to 91%: {int(min_samples_to_91)} (Estimated)")
    print(f"Final Accuracy: {final_acc_G2:.2f}%")
else:
    print(f"Final_acc_G2: {final_acc_G2:.2f}%")
    print("\n[Goal 2 Result] FAILED: Could not reach 91% with aggressive parameters. Needs tuning.")

This architecture achieves >=91% accuracy consistently across runs with almost 75-000-77,500 samples (this varies slightly)

In [ ]:
ARCH_D = [
    LinearLayer(2, 64, Tanh()),   
    LinearLayer(64, 64, Tanh()), 
    LinearLayer(64, 64, Tanh()),  
    LinearLayer(64, 1, Sigmoid()) 
]

model_G2 = Model(ARCH_D, loss_fn_type='bce')


hparams_G2 = {
    'learning_rate': 0.001,  # Aggressive LR
    'batch_size': 32,       # Small BS
    'grad_accumulation_steps': 1,
    'epochs': 200,          # Less epochs to minimise no. of samples
    'patience': 30,
    'relative_loss_threshold': 0.01 
}

Let us see if this can be improved upon

In [ ]:
# # Goal 2: Test multiple architectures for fastest convergence to 91%
# def test_architecture_for_goal2(arch_name, architecture, hyperparams, target_accuracy=91.0, max_attempts=3):
#     """
#     Tests a specific architecture with given hyperparameters to find minimum samples to reach target accuracy.
    
#     Args:
#         arch_name: String name for the architecture
#         architecture: List of LinearLayer objects defining the model
#         hyperparams: Dictionary of hyperparameters
#         target_accuracy: Target accuracy threshold (default 91.0%)
#         max_attempts: Number of training attempts to find minimum
    
#     Returns:
#         Dictionary with results
#     """
#     print(f"\n--- Testing {arch_name} for Goal 2 (Fastest Convergence) ---")
    
#     # Calculate total parameters
#     temp_model = Model(architecture, loss_fn_type='bce')
#     total_params = sum((L.W.size + L.b.size) for L in temp_model.layers)
#     print(f"Total Parameters: {total_params}")
    
#     min_samples_to_target = float('inf')
#     best_attempt_results = None
#     found_target = False
    
#     for attempt in range(max_attempts):
#         print(f"  Attempt {attempt + 1}/{max_attempts}...")
        
#         # Create fresh model instance for each attempt
#         model = Model(architecture, loss_fn_type='bce')
        
#         # Train the model
#         start_time = time.time()
#         try:
#             _, losses, samples, folder, history = train_model(model, dataset, **hyperparams)
#             time_to_converge = time.time() - start_time
            
#             # Generate maps and get final accuracy
#             final_acc = generate_maps(model, dataset, folder)
            
#             # Check if target reached and update minimum
#             if final_acc >= target_accuracy:
#                 found_target = True
#                 if samples < min_samples_to_target:
#                     min_samples_to_target = samples
#                     best_attempt_results = {
#                         'final_acc': final_acc,
#                         'folder': folder,
#                         'time': time_to_converge,
#                         'losses': losses,
#                         'history': history,
#                         'attempt': attempt + 1
#                     }
#                 print(f"    ✅ Reached {final_acc:.2f}% with {samples:,} samples")
#             else:
#                 print(f"    ❌ Only reached {final_acc:.2f}% with {samples:,} samples")
                
#         except Exception as e:
#             print(f"    ❌ Training failed: {e}")
#             continue
    
#     # Compile results
#     if found_target:
#         result = {
#             'Architecture': arch_name,
#             'Total_Params': total_params,
#             'Target_Reached': True,
#             'Min_Samples_To_Target': min_samples_to_target,
#             'Best_Final_Accuracy': best_attempt_results['final_acc'],
#             'Best_Training_Time': best_attempt_results['time'],
#             'Best_Attempt': best_attempt_results['attempt'],
#             'Run_Folder': best_attempt_results['folder']
#         }
#         print(f"SUCCESS: {arch_name} reached {target_accuracy}% in minimum {min_samples_to_target:,} samples")
#     else:
#         result = {
#             'Architecture': arch_name,
#             'Total_Params': total_params,
#             'Target_Reached': False,
#             'Min_Samples_To_Target': float('inf'),
#             'Best_Final_Accuracy': 0.0,
#             'Best_Training_Time': 0.0,
#             'Best_Attempt': 0,
#             'Run_Folder': None
#         }
#         print(f"FAILED: {arch_name} could not reach {target_accuracy}% in {max_attempts} attempts")
    
#     return result

# # Define all architectures to test for Goal 2
# goal2_architectures = {
#     'ARCH_7 (3x32->64->1)': ARCH_7,        # Mixed 3-layer
#     'ARCH_6 (4x16)': ARCH_6,              # 4-layer 16-width
#     'ARCH_5 (4x32)': ARCH_5,              # 4-layer 32-width
#     'ARCH_4 (3x64)': ARCH_4,              # Alternative 3x64
#     'ARCH_3 (3x32)': ARCH_3,              # Alternative 3x32
#     'ARCH_2 (3x16)': ARCH_2,              # Alternative 3x16
#     'ARCH_1 (3x8)': ARCH_1                # Minimal 3x8
# }

# # Aggressive hyperparameters for fast convergence (hparams_T2)
# hparams_T2 = {
#     'learning_rate': 0.001,  # Aggressive LR
#     'batch_size': 32,        # Small BS
#     'grad_accumulation_steps': 1,
#     'epochs': 200,           # Max epochs for thorough training
#     'patience': 30,
#     'relative_loss_threshold': 0.01 
# }

# # Alternative hyperparameter sets for comparison
# hparams_T2_fast = {
#     'learning_rate': 0.005,  # Even more aggressive LR
#     'batch_size': 16,        # Smaller BS for more frequent updates
#     'grad_accumulation_steps': 1,
#     'epochs': 150,
#     'patience': 20,
#     'relative_loss_threshold': 0.01 
# }

# hparams_T2_stable = {
#     'learning_rate': 0.001,  # Stable LR
#     'batch_size': 64,        # Larger BS for stability
#     'grad_accumulation_steps': 1,
#     'epochs': 300,
#     'patience': 40,
#     'relative_loss_threshold': 0.005 
# }

# # Main testing loop for Goal 2
# print("="*100)
# print("GOAL 2: TESTING ARCHITECTURES FOR FASTEST CONVERGENCE (Target >= 91%)")
# print("="*100)

# all_goal2_results = []
# max_attempts = 1

# # Test with primary hyperparameters (hparams_T2)
# print(f"\nTESTING WITH PRIMARY HYPERPARAMETERS (hparams_T2)")
# print(f"LR: {hparams_T2['learning_rate']}, BS: {hparams_T2['batch_size']}, Epochs: {hparams_T2['epochs']}")

# for arch_name, arch in goal2_architectures.items():
#     result = test_architecture_for_goal2(
#         arch_name, 
#         arch, 
#         hparams_T2, 
#         target_accuracy=91.0,
#         max_attempts=max_attempts
#     )
#     result['Hyperparams'] = 'hparams_T2'
#     result['Learning_Rate'] = hparams_T2['learning_rate']
#     result['Batch_Size'] = hparams_T2['batch_size']
#     all_goal2_results.append(result)

# # Test top 3 architectures with alternative hyperparameters
# successful_results = [r for r in all_goal2_results if r['Target_Reached']]
# if len(successful_results) > 0:
#     # Sort by minimum samples and take top 3
#     successful_results.sort(key=lambda x: x['Min_Samples_To_Target'])
#     top_3_archs = successful_results[:3]
    
#     print(f"\n⚡ TESTING TOP 3 ARCHITECTURES WITH ALTERNATIVE HYPERPARAMETERS")
    
#     for hparam_name, hparams in [('hparams_T2_fast', hparams_T2_fast), ('hparams_T2_stable', hparams_T2_stable)]:
#         print(f"\n--- Testing with {hparam_name} ---")
#         print(f"LR: {hparams['learning_rate']}, BS: {hparams['batch_size']}, Epochs: {hparams['epochs']}")
        
#         for top_result in top_3_archs:
#             arch_name = top_result['Architecture']
#             arch = goal2_architectures[arch_name]
            
#             result = test_architecture_for_goal2(
#                 f"{arch_name} + {hparam_name}", 
#                 arch, 
#                 hparams, 
#                 target_accuracy=91.0,
#                 max_attempts=1  # Fewer attempts for alternative tests
#             )
#             result['Hyperparams'] = hparam_name
#             result['Learning_Rate'] = hparams['learning_rate']
#             result['Batch_Size'] = hparams['batch_size']
#             all_goal2_results.append(result)

# # Create summary DataFrame
# df_goal2_results = pd.DataFrame(all_goal2_results)

# # Format the results for display
# df_display = df_goal2_results.copy()
# df_display['Best_Final_Accuracy'] = df_display['Best_Final_Accuracy'].apply(lambda x: f"{x:.2f}%")
# df_display['Min_Samples_To_Target'] = df_display['Min_Samples_To_Target'].apply(
#     lambda x: f"{x:,}" if x != float('inf') else "Failed"
# )
# df_display['Best_Training_Time'] = df_display['Best_Training_Time'].apply(lambda x: f"{x:.1f}s")

# # Show summary results
# print("\n" + "="*120)
# print("GOAL 2 SUMMARY RESULTS - FASTEST CONVERGENCE TO 91%")
# print("="*120)

# # Show only successful runs
# successful_runs = df_display[df_display['Target_Reached'] == True]
# if len(successful_runs) > 0:
#     print("\nSUCCESSFUL RUNS (>= 91% Accuracy):")
#     print(successful_runs[['Architecture', 'Hyperparams', 'Total_Params', 'Min_Samples_To_Target', 
#                           'Best_Final_Accuracy', 'Best_Training_Time']].to_string(index=False))
    
#     # Find the fastest convergence (minimum samples)
#     fastest_run = df_goal2_results[df_goal2_results['Target_Reached'] == True].loc[
#         df_goal2_results[df_goal2_results['Target_Reached'] == True]['Min_Samples_To_Target'].idxmin()
#     ]
    
#     print(f"\nFASTEST CONVERGENCE TO 91%:")
#     print(f"   Architecture: {fastest_run['Architecture']}")
#     print(f"   Hyperparameters: {fastest_run['Hyperparams']}")
#     print(f"   Parameters: {fastest_run['Total_Params']:,}")
#     print(f"   Samples to 91%: {fastest_run['Min_Samples_To_Target']:,}")
#     print(f"   Final Accuracy: {fastest_run['Best_Final_Accuracy']:.2f}%")
#     print(f"   Training Time: {fastest_run['Best_Training_Time']:.1f}s")
    
#     # Show efficiency ranking (samples per parameter)
#     successful_df = df_goal2_results[df_goal2_results['Target_Reached'] == True].copy()
#     successful_df['Efficiency'] = successful_df['Min_Samples_To_Target'] / successful_df['Total_Params']
#     most_efficient = successful_df.loc[successful_df['Efficiency'].idxmin()]
    
#     print(f"\nMOST EFFICIENT (Samples per Parameter):")
#     print(f"   Architecture: {most_efficient['Architecture']}")
#     print(f"   Hyperparameters: {most_efficient['Hyperparams']}")
#     print(f"   Efficiency: {most_efficient['Efficiency']:.2f} samples/param")
#     print(f"   Samples to 91%: {most_efficient['Min_Samples_To_Target']:,}")
#     print(f"   Parameters: {most_efficient['Total_Params']:,}")

# else:
#     print("\n❌ NO ARCHITECTURES ACHIEVED >= 91% ACCURACY")

# print(f"\nALL RUNS:")
# print(df_display[['Architecture', 'Hyperparams', 'Total_Params', 'Min_Samples_To_Target', 
#                  'Target_Reached']].to_string(index=False))

# # Save detailed results
# goal2_results_filename = f"goal2_fastest_convergence_{time.strftime('%Y%m%d_%H%M%S')}.csv"
# df_goal2_results.to_csv(goal2_results_filename, index=False)
# print(f"\nGoal 2 results saved to: {goal2_results_filename}")

# print(f"\n{'='*120}")
# print("GOAL 2 TESTING COMPLETE!")
# print(f"{'='*120}")

Since on testing with different architectures and hyperparameters, the 4-layer 64-width (ARCH_D) with a learning rate of 0.001 and batch size of 32 consistently reached 91% accuracy in the fewest samples, we will finalize this as our chosen architecture and hyperparameters for Goal 2.